In [ ]:
import pandas as pd
import os
import cv2
import numpy as np
import joblib
from sklearn.metrics import accuracy_score


base_path = '/mnt/d/malak/sss_ai/face-recognition-6/test'
csv_path = os.path.join(base_path, '_classes.csv')


df_classes = pd.read_csv(csv_path)
df_classes.columns = df_classes.columns.str.strip()
clf = joblib.load('face_svm_model.pkl')

y_true, y_pred, confidences = [], [], []

print(f"🚀 Analyzing {len(df_classes)} images...")

for index, row in df_classes.iterrows():
    img_name = row['filename'].strip()
    img_path = os.path.join(base_path, img_name)
    
    if os.path.exists(img_path):
        img = cv2.imread(img_path)
        if img is not None:
           
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_resized = cv2.resize(img_rgb, (224, 224))
            img_array = np.expand_dims(img_resized / 255.0, axis=0)
            
            
            embedding = feature_extractor.predict(img_array, verbose=0)
            probs = clf.predict_proba(embedding)
            
           
            y_true.append(np.argmax(row[1:].values))
            y_pred.append(np.argmax(probs))
            confidences.append(np.max(probs))


print(f"✅ Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")
print(f"🛡️ Avg Confidence: {np.mean(confidences)*100:.2f}%")


df_classes['Confidence'] = [f"{c*100:.2f}%" for c in confidences]

df_classes.to_csv(os.path.join(base_path, 'analysis_results.csv'), index=False)

🚀 Analyzing 149 images...
✅ Accuracy: 100.00%
🛡️ Avg Confidence: 99.14%
